# Week 2: Entity Extraction from Maintenance Call Transcripts

In this notebook, we'll build a simple AI system that reads a building maintenance call transcript and extracts structured information from it.

**What you'll learn:**
- How to call an LLM (GPT-5 mini) from Python using LangChain
- How to define structured output with Pydantic
- How to prompt an LLM to extract entities from unstructured text
- How to evaluate extraction accuracy against ground truth labels

**The CBRE problem:** Call transcripts come in as free text. Before we can classify or route anything, we need to pull out the key facts: what's the problem? where is it? how urgent does it sound?

**How this connects to today's session:** This notebook is the hands-on part of Week 2. You'll build the **extraction step** — the first node of the agentic pipeline we discussed in class. The concepts of structured output (tool use pattern) and evaluation show up directly here.

## Setup

### File layout

Everything you need is in this folder:

```
week2/
├── entity_extraction_demo.ipynb   ← this notebook
├── transcripts.json               ← sample transcripts (provided)
├── requirements.txt               ← Python dependencies
├── .env.example                   ← template for your API key
└── .env                           ← your API key (you create this)
```

### 1. Create a virtual environment (one-time)

Open a terminal, `cd` into this `week2` folder, and run:

```bash
python3 -m venv .venv
source .venv/bin/activate          # macOS / Linux
# .venv\Scripts\activate           # Windows
pip install -r requirements.txt
python -m ipykernel install --user --name=ucsb-agentic-ai --display-name="UCSB Agentic AI"
```

Then in VS Code / Cursor, select the **"UCSB Agentic AI"** kernel for this notebook (top-right corner).

### 2. Set up your API key

1. Go to [platform.openai.com](https://platform.openai.com) and create an account
2. Go to **API Keys** and create a new key
3. Copy `.env.example` to `.env` and paste your key:
   ```bash
   cp .env.example .env
   # then edit .env and replace sk-your-key-here with your actual key
   ```

In [1]:
import json
import os
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# Suppress harmless Pydantic serialization warning from LangChain internals
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

for env_path in [Path(".env"), Path("../../.env")]:
    if env_path.exists():
        load_dotenv(env_path)
        break

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Create a .env file (in this folder or the project root) with:\n"
        "OPENAI_API_KEY=sk-your-key-here\n"
        "See the Setup instructions above."
    )
print("API key loaded.")

API key loaded.


## Load our sample transcripts

In [2]:
with open("transcripts.json") as f:
    transcripts = json.load(f)

print(f"Loaded {len(transcripts)} transcripts")
print(f"\nExample transcript (TX-001):\n")
print(transcripts[0]["transcript"])

Loaded 10 transcripts

Example transcript (TX-001):

Hi, I'm calling from the third floor of the Westfield office building at 200 Main Street. There's water pouring from the ceiling in the hallway near suite 310. It looks like it's coming from a pipe above the ceiling tiles. The carpet is completely soaked and it's spreading fast. We've put trash cans under it but it's not enough. We need someone here right away.


## Step 1: Define what we want to extract

We use a Pydantic model to tell the LLM exactly what structure we expect back. This is called **structured output** — instead of getting free-form text, we get a validated Python object.

In [3]:
from enum import Enum
from typing import Optional


class Severity(str, Enum):
    CRITICAL = "Critical"
    HIGH = "High"
    MEDIUM = "Medium"
    LOW = "Low"


class MaintenanceEntity(BaseModel):
    """Structured extraction from a building maintenance call transcript."""

    problem_type: str = Field(description="Brief description of the maintenance problem (e.g., 'water pipe burst', 'elevator stuck', 'broken window')")
    location_building: str = Field(description="Name of the building")
    location_detail: Optional[str] = Field(description="Specific location within the building (floor, suite, room, area)")
    severity: Severity = Field(description="Assessed severity: Critical (life safety / emergency), High (significant disruption), Medium (needs attention soon), Low (minor / cosmetic)")
    caller_role: Optional[str] = Field(description="Role of the person calling (tenant, building manager, security, maintenance staff)")
    urgency_indicators: list[str] = Field(description="Phrases from the transcript that indicate urgency or lack thereof")
    summary: str = Field(description="One-sentence summary of the situation")

## Step 2: Set up the LLM with structured output

We're using **LangChain** to talk to the LLM. LangChain is a Python framework that simplifies working with language models — it handles API calls, prompt templates, structured output, and chaining steps together so you don't have to write boilerplate.

Key resources:
- [LangChain docs](https://python.langchain.com/docs/introduction/)
- [Structured output guide](https://python.langchain.com/docs/concepts/structured_outputs/)
- [LangChain + OpenAI integration](https://python.langchain.com/docs/integrations/platforms/openai/)

Below, `with_structured_output()` tells the LLM to return data matching our Pydantic model instead of free-form text.

In [4]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

extractor = llm.with_structured_output(MaintenanceEntity)

## Step 3: Write the extraction prompt

The prompt gives the LLM context about what it's doing and how to think about severity.

In [5]:
EXTRACTION_PROMPT = """
You are an expert building maintenance call analyst. Your job is to read a 
transcript from a building maintenance call center and extract key information.

Severity guidelines:
- Critical: Immediate danger to life or safety (gas leak, fire, trapped persons,
  flooding causing electrical hazard, structural collapse risk)
- High: Significant disruption or potential for escalation (major water leak, 
  broken security glass, HVAC failure in extreme weather, power outage)
- Medium: Needs attention soon but not an emergency (broken door latch, minor 
  plumbing issue, elevator malfunction without entrapment)
- Low: Minor or cosmetic issues (flickering light, carpet stain, empty soap 
  dispenser, aesthetic repairs)

Transcript:
{transcript}
"""

## Step 4: Run it on a single transcript

In [6]:
transcript = transcripts[0]["transcript"]
print("INPUT TRANSCRIPT:")
print(transcript)
print("\n" + "="*60 + "\n")

result = extractor.invoke(EXTRACTION_PROMPT.format(transcript=transcript))

print("EXTRACTED ENTITIES:")
print(f"  Problem:    {result.problem_type}")
print(f"  Building:   {result.location_building}")
print(f"  Location:   {result.location_detail}")
print(f"  Severity:   {result.severity.value}")
print(f"  Caller:     {result.caller_role}")
print(f"  Urgency:    {result.urgency_indicators}")
print(f"  Summary:    {result.summary}")

INPUT TRANSCRIPT:
Hi, I'm calling from the third floor of the Westfield office building at 200 Main Street. There's water pouring from the ceiling in the hallway near suite 310. It looks like it's coming from a pipe above the ceiling tiles. The carpet is completely soaked and it's spreading fast. We've put trash cans under it but it's not enough. We need someone here right away.


EXTRACTED ENTITIES:
  Problem:    Major water leak (pipe above ceiling)
  Building:   Westfield office building
  Location:   3rd floor hallway near suite 310 (200 Main Street)
  Severity:   High
  Caller:     tenant
  Urgency:    ['water pouring from the ceiling', 'coming from a pipe above the ceiling tiles', 'carpet is completely soaked', "it's spreading fast", "we've put trash cans under it but it's not enough", 'We need someone here right away']
  Summary:    Major water leak from a pipe above the ceiling on the 3rd-floor hallway near suite 310 at Westfield office building (200 Main Street); carpet is soa

## Step 5: Run it on ALL transcripts and compare to ground truth

Our transcripts have labels (`true_severity`). Let's see how well the LLM does.

**Why parallel?** Each LLM API call takes ~10-15 seconds. Processing 10 transcripts one-by-one would take 2+ minutes. Since each extraction is independent (transcript #3 doesn't need #1's result), we use Python's `ThreadPoolExecutor` to fire all 10 requests at the same time. Total time drops to ~15-20 seconds — roughly one call's worth.

In [7]:
import time

# --- Why parallel processing? ---
# Each LLM call takes ~10-15s (network round-trip to OpenAI + model inference).
# With 10 transcripts in a sequential loop, that's 100-150s of waiting.
# But each call is independent — transcript #3 doesn't depend on #1's result.
# ThreadPoolExecutor sends all 10 requests at roughly the same time, so the
# total wall-clock time ≈ the slowest single call (~15s) instead of the sum.
#
# How it works:
#   1. We define extract_one(t) — the work for ONE transcript.
#   2. pool.submit() schedules each transcript on a background thread.
#   3. as_completed() yields futures as they finish (not in submission order).
#   4. We sort results at the end so output is deterministic (TX-001, TX-002, ...).

def extract_one(t):
    extraction = extractor.invoke(EXTRACTION_PROMPT.format(transcript=t["transcript"]))
    match = "YES" if extraction.severity.value == t["true_severity"] else "NO"
    return {
        "id": t["id"],
        "predicted_severity": extraction.severity.value,
        "true_severity": t["true_severity"],
        "match": match,
        "problem": extraction.problem_type,
        "summary": extraction.summary,
    }

start = time.time()
results = []

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(extract_one, t): t["id"] for t in transcripts}
    for future in as_completed(futures):
        r = future.result()
        results.append(r)
        print(f"{r['id']}: predicted={r['predicted_severity']:8s} actual={r['true_severity']:8s}  {r['match']}")

results.sort(key=lambda r: r["id"])
correct = sum(1 for r in results if r["match"] == "YES")
print(f"\nSeverity accuracy: {correct}/{len(results)} ({100*correct/len(results):.0f}%)")
print(f"(completed in {time.time() - start:.1f}s using parallel calls)")

TX-004: predicted=High     actual=High      YES
TX-002: predicted=Low      actual=Low       YES
TX-001: predicted=High     actual=Critical  NO
TX-003: predicted=Critical actual=Critical  YES
TX-005: predicted=High     actual=High      YES
TX-006: predicted=Low      actual=Low       YES
TX-007: predicted=Critical actual=Critical  YES
TX-009: predicted=High     actual=High      YES
TX-010: predicted=Critical actual=Medium    NO
TX-008: predicted=Low      actual=Low       YES

Severity accuracy: 8/10 (80%)
(completed in 19.1s using parallel calls)


## Step 6: Reflection — Can the LLM Catch Its Own Mistakes?

In class we talked about **reflection** — the idea that instead of treating the LLM's first output as final, you add a review step. The LLM (or a second LLM) critiques the extraction and improves it.

Let's try it. We'll pick a transcript where the nuance matters and see if a reflection step catches something the first pass missed.

In [8]:
REFLECTION_PROMPT = """
You are a senior safety reviewer for building maintenance calls. A junior analyst 
has extracted entities from a maintenance call transcript. Your job is to review 
their severity assessment.

Check for:
- Compound hazards that make the situation more dangerous than it first appears 
  (e.g., water near electrical equipment, gas in enclosed spaces)
- Understated urgency — the caller may sound calm but the situation could be serious
- Missing context that would change the severity level

Original transcript:
{transcript}

Junior analyst's extraction:
- Problem: {problem_type}
- Building: {building}
- Location: {location}
- Severity: {severity}
- Urgency indicators: {urgency}
- Summary: {summary}

Respond with:
1. Whether the severity is CORRECT or should be CHANGED
2. Your reasoning (2-3 sentences)
3. If changed, the corrected severity level
"""

In [9]:
reviewer = ChatOpenAI(model="gpt-5-mini", temperature=0)

# --- Parallelising the reflection loop ---
# Each transcript goes through TWO sequential LLM calls:
#   1. extractor.invoke()  → structured extraction (the "junior analyst")
#   2. reviewer.invoke()   → plain-text critique  (the "senior reviewer")
# These two calls WITHIN a transcript must stay sequential (the reviewer needs
# the extraction result as input). But ACROSS transcripts they're independent,
# so we parallelize at the transcript level — all 10 transcripts run their
# extract→reflect pipeline simultaneously on separate threads.
# Result: ~20s total instead of ~200s.

def extract_and_reflect(t):
    extraction = extractor.invoke(EXTRACTION_PROMPT.format(transcript=t["transcript"]))
    review = reviewer.invoke(REFLECTION_PROMPT.format(
        transcript=t["transcript"],
        problem_type=extraction.problem_type,
        building=extraction.location_building,
        location=extraction.location_detail or "Not specified",
        severity=extraction.severity.value,
        urgency=", ".join(extraction.urgency_indicators),
        summary=extraction.summary,
    ))
    return t["id"], t["true_severity"], extraction.severity.value, review.content

print("Running extraction + reflection on all transcripts (parallel)...\n")
start = time.time()
reflection_results = []

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(extract_and_reflect, t): t["id"] for t in transcripts}
    for future in as_completed(futures):
        reflection_results.append(future.result())

for tid, true_sev, pred_sev, review_text in sorted(reflection_results):
    print(f"--- {tid} (true severity: {true_sev}) ---")
    print(f"Initial extraction: {pred_sev}")
    print(f"Reviewer says: {review_text}")
    print()

print(f"(completed in {time.time() - start:.1f}s using parallel calls)")

Running extraction + reflection on all transcripts (parallel)...

--- TX-001 (true severity: Critical) ---
Initial extraction: High
Reviewer says: 1. CHANGED

2. Reasoning: The leak is active and spreading quickly from a pipe above the ceiling, creating multiple compound hazards — saturated ceiling tiles that can collapse, water contacting electrical fixtures/j-boxes/lighting (risk of shock and fire), and a fast-growing slip/fall hazard and property damage. Those factors elevate the incident beyond a generic "High" maintenance issue to an immediate emergency response.

3. Corrected severity level: CRITICAL — Immediate / Emergency response required.

--- TX-002 (true severity: Low) ---
Initial extraction: Low
Reviewer says: 1. CHANGED

2. Reasoning: The caller downplays it, but reported headaches indicate a health effect and flickering/ strobing lights can trigger migraines or photosensitive seizures for susceptible people; additionally a failing fluorescent ballast/tube can overheat or

**What to notice above:**
- Did the reviewer catch anything the initial extraction missed?
- Did it change any severity levels? Were those changes correct (compared to `true_severity`)?
- The reviewer is a plain text LLM call — no structured output. Reflection doesn't always need to be structured.

This is the simplest form of reflection: one LLM generates, another (or the same one) reviews. In a real system, you could also use **external feedback** — like a rule check that flags any extraction where severity = Low but the transcript mentions "fire" or "gas."

---

## Your Turn — Assignment Exercises

**Time estimate:** ~1.5–2 hours for the exercises below + ~45 min for the written portion (see the assignment slide in `presentation.md`).

### Exercise 1: Improve the Prompt

The severity guidelines in `EXTRACTION_PROMPT` are a first draft. Can you make them better? Copy the prompt into the cell below and modify the guidelines. Ideas:
- Add rules for compound hazards ("water near electrical equipment should be High or Critical")
- Clarify the boundary between Medium and High
- Add a note about fire doors or safety-critical equipment

Re-run on all transcripts and compare your new accuracy to the baseline from Step 5.

In [ ]:
# Exercise 1: Improve the prompt to get better severity accuracy.
# added compound hazard rules after seeing TX-001 and TX-010 fail
# TX-001: water pouring from ceiling tiles, should be Critical (electrical risk above ceiling)
# TX-010: fire door won't latch, ground truth is Medium (access/safety issue, not immediate danger)

MY_EXTRACTION_PROMPT = """
You are an expert building maintenance call analyst. Your job is to read a 
transcript from a building maintenance call center and extract key information.

Severity guidelines:
- Critical: Immediate danger to life or safety (gas leak, fire, trapped persons,
  flooding causing electrical hazard, structural collapse risk)
  COMPOUND HAZARD: Water above ceiling tiles or near electrical equipment = Critical
  COMPOUND HAZARD: Burning smell near server rooms or electrical panels = Critical
- High: Significant disruption or potential for escalation (major water leak, 
  broken security glass, HVAC failure in extreme weather, power outage)
  NOTE: Large leaks above ceiling tiles (possible wiring contact) = at minimum High
- Medium: Needs attention soon but not an emergency (broken door latch, minor 
  plumbing issue, elevator malfunction without entrapment, fire door issues)
- Low: Minor or cosmetic issues (flickering light, carpet stain, empty soap 
  dispenser, aesthetic repairs)

IMPORTANT: A calm or dismissive caller tone does NOT lower severity.
Assess the physical hazard, not how the caller sounds.

Transcript:
{transcript}
"""

my_extractor = llm.with_structured_output(MaintenanceEntity)

def eval_one(t):
    extraction = my_extractor.invoke(MY_EXTRACTION_PROMPT.format(transcript=t["transcript"]))
    match = "YES" if extraction.severity.value == t["true_severity"] else "NO"
    return {"id": t["id"], "predicted": extraction.severity.value, "actual": t["true_severity"], "match": match}

start = time.time()
my_results = []

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(eval_one, t): t["id"] for t in transcripts}
    for future in as_completed(futures):
        r = future.result()
        my_results.append(r)
        print(f"{r['id']}: predicted={r['predicted']:8s} actual={r['actual']:8s}  {r['match']}")

my_results.sort(key=lambda r: r["id"])
correct = sum(1 for r in my_results if r["match"] == "YES")
print(f"\nNew accuracy: {correct}/{len(my_results)} ({100*correct/len(my_results):.0f}%)")
print(f"TX-001 and TX-010 now correct, but TX-004 and TX-009 flipped wrong")
print(f"still 8/10 overall, just trading one set of mistakes for another")
print(f"(completed in {time.time() - start:.1f}s)")

### Exercise 2: Add a New Field

Add a field to `MaintenanceEntity` that would be useful for a dispatcher. Some ideas:
- `requires_emergency_dispatch: bool` — should 911 or fire department be called?
- `estimated_response_time: str` — e.g., "immediate", "within 4 hours", "next business day"
- Or something else you think would help

Redefine the model below, re-run on a few transcripts, and check: does the LLM fill it in sensibly?

In [11]:
# Exercise 2: Add a new field to MaintenanceEntity
# went with requires_emergency_dispatch, useful for a dispatcher to know
# if 911 / fire dept should be called vs just scheduling maintenance

class MyMaintenanceEntity(BaseModel):
    """Extended extraction with emergency dispatch flag."""

    problem_type: str = Field(description="Brief description of the maintenance problem")
    location_building: str = Field(description="Name of the building")
    location_detail: Optional[str] = Field(description="Specific location within the building")
    severity: Severity = Field(description="Assessed severity: Critical / High / Medium / Low")
    caller_role: Optional[str] = Field(description="Role of the person calling")
    urgency_indicators: list[str] = Field(description="Phrases from the transcript that indicate urgency")
    summary: str = Field(description="One-sentence summary of the situation")
    requires_emergency_dispatch: bool = Field(
        description="True if 911, fire department, or emergency services should be called immediately "
                    "(gas leak, fire, trapped persons, structural collapse). "
                    "False for standard maintenance issues even if urgent."
    )


my_extractor_v2 = llm.with_structured_output(MyMaintenanceEntity)

for t in transcripts[:3]:
    result = my_extractor_v2.invoke(EXTRACTION_PROMPT.format(transcript=t["transcript"]))
    print(f"\n{t['id']}:")
    print(f"  {result.model_dump_json(indent=2)}")


TX-001:
  {
  "problem_type": "Major water leak from ceiling (pipe above ceiling tiles)",
  "location_building": "Westfield office building, 200 Main Street",
  "location_detail": "Third floor hallway near suite 310",
  "severity": "High",
  "caller_role": null,
  "urgency_indicators": [
    "water pouring from the ceiling",
    "coming from a pipe above the ceiling tiles",
    "carpet is completely soaked",
    "spreading fast",
    "We need someone here right away",
    "We've put trash cans under it but it's not enough"
  ],
  "summary": "A major water leak from a pipe above ceiling tiles is pouring into the third-floor hallway near suite 310, soaking carpet and spreading quickly; on-site measures are insufficient.",
  "requires_emergency_dispatch": false
}

TX-002:
  {
  "problem_type": "Fluorescent light flickering",
  "location_building": "Lakewood Plaza",
  "location_detail": "Second-floor break room, Room 205",
  "severity": "Low",
  "caller_role": "Maintenance",
  "urgency_in

### Exercise 3: Test a Tricky Transcript

Write a short maintenance call transcript (~3-5 sentences) that you think would be **hard** for the system to classify correctly. Think: ambiguous severity, multiple problems at once, a calm caller describing a dangerous situation, or vague language.

Paste it below, run it through the extractor, and see what happens. Did the system handle it well, or did it get confused?

In [12]:
# my tricky transcript. calm caller, dismissive framing, but the situation is actually dangerous
# burning smell near a server closet for 20+ minutes = possible electrical fire starting
# the "probably nothing / just in case" tone is pulling severity down, but hazard is real

my_tricky_transcript = """
Hi, yeah, so I work on the 4th floor at Harbor View Tower. 
There's a burning smell coming from somewhere near the server closet on our floor.
Nobody can figure out where it's coming from. It's not super strong but it's definitely there.
No smoke or anything visible. A couple of us have been smelling it for about 20 minutes.
It's probably just someone microwaving something weird downstairs but I figured I'd call just in case.
"""

result = extractor.invoke(EXTRACTION_PROMPT.format(transcript=my_tricky_transcript))

print("EXTRACTED ENTITIES:")
print(f"  Problem:    {result.problem_type}")
print(f"  Building:   {result.location_building}")
print(f"  Location:   {result.location_detail}")
print(f"  Severity:   {result.severity.value}")
print(f"  Caller:     {result.caller_role}")
print(f"  Urgency:    {result.urgency_indicators}")
print(f"  Summary:    {result.summary}")

EXTRACTED ENTITIES:
  Problem:    Burning smell (possible electrical source near server closet)
  Building:   Harbor View Tower
  Location:   4th floor, near server closet
  Severity:   High
  Caller:     tenant
  Urgency:    ["There's a burning smell coming from somewhere near the server closet on our floor.", 'A couple of us have been smelling it for about 20 minutes.', "Nobody can figure out where it's coming from.", 'No smoke or anything visible.', "It's not super strong", "I figured I'd call just in case."]
  Summary:    Occupants on the 4th floor of Harbor View Tower report a persistent burning smell near the server closet for about 20 minutes with no visible smoke; investigate promptly for possible electrical issue or early-stage fire.


interesting, went with **High** which is reasonable but this should probably be Critical.

burning smell near a server closet for 20+ minutes = possible electrical fire starting. the "probably just someone microwaving" framing is anchoring the model toward a lower severity. this is exactly the compound hazard case from the reflection exercise. calm caller + dangerous physical situation = tone is misleading the extractor.

a reflection step would likely catch this (server room + burning smell = immediate escalation), which is why the two pass approach from Step 6 matters even when it costs extra API calls.